## BBM 409 - Programming Assignment 1

###  Mustafa Ege 2210356088

## Understanding Perceptron Learning Algorithm (100 points)

### 1. Data Loading and Exploration (5 points)
* Brief overview of the task.
* Description of the dataset used for analysis.

In [ ]:
## Import necessary libraries
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split 

In [ ]:
# fetch dataset
raisin = fetch_ucirepo(id=850)

# data (as pandas dataframes)
X_df = raisin.data.features
y_df = raisin.data.targets

# metadata
print(raisin.metadata)

# variable information
print(raisin.variables)

In [ ]:
# Preview first few rows
print("\nFirst 5 rows:")
print(X_df.head())

# Dataset info
print("\nDataset info:")
print(X_df.info())

# Summary statistics
print("\nSummary statistics:")
print(X_df.describe())

# Unique class labels
print("\nTarget variable distribution:")
print(y_df['Class'].value_counts())

# Check for missing values
print("\nMissing values per column:")
print(X_df.isnull().sum())


Convert the data frame into a Numpy array.

In [ ]:
X_np = X_df.to_numpy()
y_np = y_df.to_numpy().ravel()  # flatten to 1D array

print("\nConverted to NumPy arrays:")
print("X shape:", X_np.shape)
print("y shape:", y_np.shape)


### 2. Data Preprocessing (5 points)
Explain the preprocessing steps taken and their rationale.

#### Feature Scaling Techniques:
Feature scaling is important for many machine learning algorithms, especially those based on distance metrics (e.g., k-nearest neighbors, support vector machines) and gradient descent optimization (e.g., linear regression, logistic regression).

Some common feature scaling techniques include:

1. **Standardization (Z-score normalization):** It scales the features to have a mean of 0 and a standard deviation of 1.
2. **Min-Max Scaling:** It scales the features to a fixed range, typically between 0 and 1.
3. **Robust Scaling:** It scales the features based on percentiles, making it robust to outliers.

You can use these scaled features for training your machine learning models.

Remember, the choice of feature selection and feature scaling techniques can significantly impact the performance of your models. It's essential to experiment and choose the techniques that work best for your specific dataset and machine learning tasks.

In [ ]:
## Analyze the dataset and decide whether scaling is needed.
## Explain your choices in a brief comment.

Standardization is needed because the features like 'Area' and 'Eccentricity' are on completely on different scales one being around thousands and the other being smaller than 1. Features with large absolute values would dominate the distance/weighted sum calculation, preventing the Perceptron from learning an optimal weight vector.

Therefore, we will standardize the data with Z-score normalization.

The labels of raisins are in categorical format as Besni and Kecimen, we first need to transform those labels into -1 and +1 for perceptron algorithm which requires class labels of -1, +1.

In [ ]:
# Perceptron requires labels in {-1, 1}. Transform y_np:
# Class 0 (Besni) -> -1
# Class 1 (Kecimen) -> +1

# Check the unique classes before transformation
print("Original unique classes:", np.unique(y_np)) 

# 1. Direct mapping from strings to Perceptron labels (-1, +1)
label_mapping = {'Besni': -1, 'Kecimen': 1} 

# 2. Apply the mapping to create the final transformed labels
# We use a list comprehension or a similar array-based method for this.
y_transformed = np.array([label_mapping[label] for label in y_np])

# Final verification
print("Transformed unique classes:", np.unique(y_transformed))

#### Training and Validation Split
1. Explain why it’s important to use the validation set to evaluate your model instead
of relying solely on the training data.
2. Discuss how evaluating the model on unseen
data helps in assessing its generalization capability.

Calculating the mean and standard deviation from the full dataset before splitting causes data leakage, therefore we first split the data into training and validation set to use the training data for calculating mean and deviation. Then we will update both sets with standardized values using the mean and deviation.

In [ ]:
# Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_np, y_transformed, test_size=0.2, random_state=23, shuffle=True
)

# Print the shapes of the resulting sets
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of Y_train:", y_train.shape) 
print("Shape of Y_test:", y_test.shape)   

In [ ]:
def standardize(X_train, X_test):
    """
    Standardize features using ONLY training set statistics (Z-score normalization).
    Returns:
      x_train_zscore_scaled, x_test_zscore_scaled, mean_train, std_train_safe
    """
    mean_train = np.mean(X_train, axis=0)
    std_train = np.std(X_train, axis=0)

    # avoid division by zero
    std_train_safe = np.where(std_train == 0, 1e-8, std_train)

    x_train_standardize = (X_train - mean_train) / std_train_safe
    x_test_standardize = (X_test - mean_train) / std_train_safe

    return x_train_standardize, x_test_standardize


In [ ]:
## Feature Scaling - Standardization (Z-score normalization)

# Use the function to standardize based on training statistics
x_train_standardize, x_test_standardize, mean_train, std_train = standardize(X_train, X_test)

# Keep original variable names for compatibility with later cells
X_train_scaled = x_train_standardize
X_test_scaled = x_test_standardize

print("\nX_train_scaled (first 3 rows):")
print(X_train_scaled[:3])
print("\nMean of X_train_scaled features (should be near 0):", np.mean(X_train_scaled, axis=0))
print("Std Dev of X_train_scaled features (should be near 1):", np.std(X_train_scaled, axis=0))

In [ ]:
def minmax_scale(X_train, X_test):
    """
    Applies Min-Max scaling to features using ONLY training set statistics.
    X_scaled = (X - X_min) / (X_max - X_min)
    """
    # 1. Calculate min and max ONLY from the training data
    min_train = np.min(X_train, axis=0)
    max_train = np.max(X_train, axis=0)
    
    # Calculate the range (denominator)
    range_train = max_train - min_train
    
    # Handle division by zero if a feature is constant (unlikely here)
    range_train[range_train == 0] = 1e-8 
    
    # 2. Apply scaling to training and test sets
    X_train_minmax_scaled = (X_train - min_train) / range_train
    X_test_minmax_scaled = (X_test - min_train) / range_train
    
    return X_train_minmax_scaled, X_test_minmax_scaled

In [ ]:
def robust_scale(X_train, X_test):
    """
    Applies Robust scaling to features using ONLY training set statistics.
    X_scaled = (X - Median) / IQR
    """
    # 1. Calculate Median (50th percentile) from the training data
    median_train = np.median(X_train, axis=0)
    
    # 2. Calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1_train = np.percentile(X_train, 25, axis=0)
    Q3_train = np.percentile(X_train, 75, axis=0)
    
    # 3. Calculate the Interquartile Range (IQR)
    IQR_train = Q3_train - Q1_train
    
    # Handle division by zero
    IQR_train[IQR_train == 0] = 1e-8
    
    # 4. Apply scaling to training and test sets
    X_train_robust_scaled = (X_train - median_train) / IQR_train
    X_test_robust_scaled = (X_test - median_train) / IQR_train
    
    return X_train_robust_scaled, X_test_robust_scaled

### 3. Implementing Perceptron for Binary Classification (20 points)

Explain the reason behind the application of perceptron learning algorithm on this type of data. If the dataset had two classes but was not linearly separable, how would the perceptron's performance change? Briefly explain.

####  Implement Percepton Learning Algorithm

 The use of basic libraries like Matlab, pandas, and numpy is allowed, but the use of ready-to-use libraries such as scikit-learn is prohibited

In [ ]:
class Perceptron:
    def __init__(self, learning_rate=0.01, n_epochs=50, random_state=23):
        # Initialize the model parameters (weights, bias) and hyperparameters
        self.lr = learning_rate
        self.n_epochs = n_epochs
        # Setting random_state for reproducibility in NumPy's random functions
        self.random_state = random_state 
        self.weights = None
        self.bias = None
        self.errors_ = []  # To store misclassification count per epoch

    def fit(self, X, y):
        rgen = np.random.RandomState(self.random_state)
        
        # Initialize weights to small random numbers, and bias to 0.0
        # X.shape[1] is the number of features
        self.weights = rgen.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.bias = 0.0
        
        # Iterate over epochs
        for _ in range(self.n_epochs):
            errors = 0
            # Iterate over each training sample
            for xi, target in zip(X, y):
              
                # Check for misclassification: 
                # The update only occurs if y_i and prediction have different signs
                prediction = self.predict(xi)
                
                if target != prediction:
                    # Target is the true label (y_i in {-1, 1})
                    # xi is the input vector

                    # Calculate prediction: 
                    update = self.lr * target # update is eta * y_i

                    # Update weights: w <- w + eta * y_i * x_i 
                    self.weights += update * xi
                    
                    # Update bias: b <- b + eta * y_i 
                    self.bias += update
                    
                    errors += 1
            
            self.errors_.append(errors)
            
            # Add an early stop if errors == 0 (convergence)
            if errors == 0:
                print("Model converged")
                break
                
        return self

    def weighted_sum(self, X):
        # Calculate the weighted sum z = w^T * x + b
        # The result must be a 1D array of z values (one for each row in X)
        return np.dot(X, self.weights) + self.bias

    def predict(self, X):
        # Determine the class label based on the sign of z (Step Function)
        return np.where(self.weighted_sum(X) >= 0.0, 1, -1)

*Explain each parameter used in your perceptron model (for example: weights, bias, learning rate, random_state, shuffle, etc.).  
For each parameter, describe in detail its role in the model’s learning process and explain why it is important.*

##### * Make sure you have defined all the necessary parameters in your perceptron model to optimize its performance.

Discuss how the choice of learning rate and the number of epochs might affect the
training process. What happens with a very small or very large learning rate?

#### Accuracy implementation

In [ ]:
def calculate_accuracy(y_true, y_pred):
    """
    Calculates the accuracy score.
    y_true: true labels (NumPy array)
    y_pred: predicted labels (NumPy array)
    """
    # Compare true labels and predicted labels. 
    # np.sum counts the number of 'True' comparisons (correct predictions).
    correct_predictions = np.sum(y_true == y_pred)
    total_samples = len(y_true)
    
    accuracy = correct_predictions / total_samples
    return accuracy

#### Train the Perceptron

In [ ]:
## Train the Perceptron model on the training set. You will use all features in the first classification.

In [ ]:
# 1. Instantiate the Perceptron model
ppn = Perceptron(learning_rate=0.0001, n_epochs=100000) # Example hyperparameters

# 2. Train the model
ppn.fit(X_train_scaled, y_train) 

# 3. Predict on training data
y_pred_train = ppn.predict(X_train_scaled)

In [ ]:
# Instantiate the Perceptron model
ppn = Perceptron() 

# Train the model on the training set
ppn.fit(X_train_scaled, y_train) 

# Predicting on the training data
y_pred_train = ppn.predict(X_train_scaled)

print("Perceptron training complete.")

#### Prediction on Training Data

In [ ]:
# Calculate the training accuracy
accuracy_train = calculate_accuracy(y_train, y_pred_train)

print(f"Training Accuracy: {accuracy_train:.4f}")

In [ ]:
# Apply standardization
X_train_standardize, X_test_standardize = standardize(X_train, X_test)

# Instantiate and train Perceptron
ppn_standardize = Perceptron() 
ppn_standardize.fit(X_train_standardize, y_train) 

# Predict and evaluate
y_pred_minmax = ppn_standardize.predict(X_train_standardize)
accuracy_minmax = calculate_accuracy(y_train, y_pred_minmax)

print(f"Min-Max Training Accuracy: {accuracy_minmax:.4f}")

In [ ]:
# Apply Min-Max Scaling
X_train_standardize, X_test_standardize = minmax_scale(X_train, X_test)

# Instantiate and train Perceptron with Min-Max scaled data
# Use your desired hyperparameters
ppn_standardize = Perceptron() 
ppn_standardize.fit(X_train_standardize, y_train) 

# Predict and evaluate
y_pred_minmax = ppn_standardize.predict(X_train_standardize)
accuracy_minmax = calculate_accuracy(y_train, y_pred_minmax)

print(f"Min-Max Training Accuracy: {accuracy_minmax:.4f}")

In [ ]:
# Apply Robust Scaling
X_train_robust, X_test_robust = robust_scale(X_train, X_test)

# Instantiate and train Perceptron with Robust scaled data
# Use your desired hyperparameters
ppn_robust = Perceptron() 
ppn_robust.fit(X_train_robust, y_train) 

# Predict and evaluate
y_pred_robust = ppn_robust.predict(X_train_robust)
accuracy_robust = calculate_accuracy(y_train, y_pred_robust)

print(f"Robust Training Accuracy: {accuracy_robust:.4f}")

Answer the Guiding Questions:
1. What does accuracy tell us about the performance of the model?
2. Why is accuracy sometimes not enough, especially in cases where the data is imbalanced? Explain a scenario where a high accuracy might be misleading.
3. In what types of problems is precision more important than recall?
4. In what cases is recall more critical?
5. When is the F1 score a better measure than accuracy? What are the limitations of using F1 score alone?

### 4. Evaluating Perceptron for Binary Classification (25 points)

#### Prediction on Validation Data

In [ ]:
## Evaluate the model's performance on the test set by calculating accuracy. Print classification report and confusion matrix

print("Testing Accuracy:", accuracy_test)

Answer the Guiding Questions:
1. What does accuracy tell us about the performance of the model?
2. Why is accuracy sometimes not enough, especially in cases where the data is imbal-
anced? Explain a scenario where a high accuracy might be misleading.

In [ ]:
def calculate_metrics(y_true, y_pred):

    return accuracy, precision, recall, f1

# Calculate and print metrics
accuracy_train, precision_train, recall_train, f1_train = calculate_metrics(y_train.flatten(), y_pred_train)

print(f"Training Accuracy: {accuracy_train}")
print(f"Training Precision: {precision_train}")
print(f"Training Recall: {recall_train}")
print(f"Training F1 Score: {f1_train}")

In [ ]:
def calculate_metrics(y_true, y_pred):
    """
    Calculates Accuracy, Precision, Recall, and F1 Score.
    The labels y_true and y_pred must be in the format {-1, +1}.
    """
    # Convert to 1D arrays if necessary
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()

    # Define the Positive class as +1, and Negative class as -1 (Besni)
    
    # Calculate the Confusion Matrix Components
    
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == -1) & (y_pred == -1))
    FP = np.sum((y_true == -1) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == -1))
    
    total_samples = len(y_true)
    
    #  Calculate Metrics 
    
    # Accuracy 
    accuracy = (TP + TN) / total_samples
    
    # Precision 
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    
    # Recall 
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    
    # F1 Score
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    # Support: The number of actual occurrences of the class (+1).
    support = TP + FN
    
    return accuracy, precision, recall, f1, support

In [ ]:
# Calculate and print metrics
accuracy_train, precision_train, recall_train, f1_train, _ = calculate_metrics(y_train.flatten(), y_pred_train)

print(f"Training Accuracy: {accuracy_train}")
print(f"Training Precision: {precision_train}")
print(f"Training Recall: {recall_train}")
print(f"Training F1 Score: {f1_train}")

 *Carefully examine the **classification report** obtained after the classification process.  
Analyze each metric in the report — such as **accuracy, precision, recall, f1-score, and support** — and interpret the performance of your model.*  

*For each metric, explain:*  
*- What this value represents in the context of your model,*  
*- What it means if the value is high or low, and*  
*- Which aspects of your model are strong or weak based on this metric.*


### 5. Visualization of the Hyperplane (40 + 5 points)

In this section, you will explore how the Perceptron Learning Algorithm forms the decision boundary (or hyperplane) that separates the two classes. To be able to do that
you will reduce the feature space to two dimensions, train the Perceptron on the selected
features, and visualize the hyperplane. The goal is to understand how the Perceptron
changes the decision boundary as the data changes, and how it adapts based on the
features used.

#### Step 1: Feature Selection Using Correlation Analysis (10 points)

1. Correlation Analysis: One effective way to reduce the number of features is by
analyzing the correlation between them. Highly correlated features may provide re-
dundant information, so you can select two less correlated features for visualization.
2. Compute the correlation matrix: Analyze the relationship between the all features to identify pairs of features with low
correlation.

In [ ]:
# Compute the correlation matrix
correlation_matrix =
# Display the correlation matrix as a heatmap

3. Choose two features: Based on the correlation analysis, select two features that are
least correlated (e.g., variance and skewness) to visualize the Perceptron’s decision
boundary.

In [ ]:
# Select two features (for example (Area and ConvexArea) and two more pair) according to their correlation
# Select these features from the original feature set
# Find a pair of features such that the separation between classes is at its highest level. Similarly, find a pair of features such that the separation between classes is at its lowest level. Finally, find the pair of features with average separability.
# Split the selected features into training and validation sets if not done yet

#### Step 2: Train the Perceptron on Selected Features (10 points)

1. Retrain the Perceptron: After selecting two features, train the Perceptron using
only these two features. Use the same training and evaluation process as before,
but now the model will work in a two-dimensional feature space.

In [ ]:
## Train the Perceptron model on the training set.

# Predicting on the training data
print("Training Accuracy:", accuracy_train)

2. Visualize the decision boundary (hyperplane): Once the Perceptron is trained, plot
the decision boundary (hyperplane) to see how the Perceptron separates the two
classes in 2D.

In [ ]:
# Function to plot the decision boundary
def plot_decision_boundary(X, y, model):

# Plot the decision boundary using the selected features and trained model

#### Step 3: Experiment with Different Features (10 points)

After visualizing the decision boundary for one pair of features, try selecting different
combinations of features (e.g., Area and MaxorAxisLength) and retrain the Perceptron. Compare
how the hyperplane changes with different features. This helps in understanding how the
Perceptron’s decision-making process adapts to different feature sets.

* Answer the guiding questions:  
1. How does the decision boundary change when you use different pairs of features?  
2. Can you find a pair of features that leads to better separation between the two classes? Also explain why this feature pair causes the better separation.  
3. If the classes cannot be linearly separated in a given pair of features, can we still talk about a linear relationship between them? Explain your reasoning.


#### Bonus Step 4: Incrementally Add Data (+ 5 points)

1. Visualizing Incremental Learning: Another experiment to help understand the behavior of the Perceptron is to add data incrementally. Start with a small portion of the dataset and progressively add more data points to train the Perceptron.

* For example, draw the decision boundary using the first 5% of the data, then increase it by 10% increments, continuing the process until the entire dataset is used, and draw the decision boundary at each step.


2. Observe how the decision boundary shifts: As you add more data, observe how the
hyperplane shifts and adapts to new points. This will help you understand how the
Perceptron continuously adjusts the boundary to separate the classes.

#### Step 6: Fisher’s Linear Discriminant Projection (10 points)
* In this step, you will implement Fisher’s Linear Discriminant (LD) to project the
dataset onto a 1D space and visualize the distribution of the two classes in the projected
space using a histogram. The goal of Fisher’s LD is to find the direction in the feature
space that maximizes the separation between two classes while minimizing the variance
within each class. This technique projects the data onto a single dimension where the
separation between the classes is optimized.

In [ ]:
def fisher_ld_projection(X, y):
    """Compute Fisher's Linear Discriminant projection direction."""

def project_data(X, direction):
    """Project data onto the Fisher LD direction."""

In [ ]:
direction = fisher_ld_projection()

# Project the data onto the Fisher LD direction
projected_data = project_data()

### Visualizations
#### 2D Feature Space (Optional but Recommended)
* While not mandatory, it can be highly instructive to visualize the projection direction computed by Fisher’s LD in the original 2D feature space. This will help you understand how the data is being projected onto a 1D space and why this direction is optimal for class separation.

In [ ]:
# Plot the 2D data with the projection direction

# Hint: You can use plt.scatter() to plot the 2D data with its features
# You can find the mean of the entire dataset for placing the direction vector
# Hint: You can use plt.quiver() method to visualize the direction vector
# Do not forget to scale the direction vector to visualize it properly
scale = 4 # Adjust this to control the length of the direction line

### 1D Projected Space (Required)
* Plot a histogram showing the distribution of the
projected data for each class in the 1D space. The histogram will help you visualize
how well the two classes are separated by the Fisher’s LD projection.

In [ ]:
# Plot the histogram of the projected data

Answer the guiding questions:
1. How well does Fisher’s LD separate the two classes in the 1D projected space?
2. Compare the separation of the two classes in the original 2D space (used in the
Perceptron) versus the 1D Fisher’s LD projection. What do you observe about the
class distributions in these spaces?